# Oracle RDF End-to-End Verification

This notebook is a manual integration test for Oracle RDF mode, aligned with the Oracle tests in `tests/driver/`.

It validates:
- Driver connectivity and pool reuse.
- RDF identifier flow from `OracleDriver` into `rdf_utils.execute_sparql_update`.
- Core Oracle RDF save/delete operations for entity nodes and entity edges.
- Query-builder expectations from `test_oracle_queries.py`.

You provide the initialized `OracleDriver` instance in the config cell.

In [ ]:
from datetime import datetime, timezone
from pprint import pprint
from uuid import uuid4

from graphiti_core.driver.driver import GraphProvider
from graphiti_core.driver.oracle.rdf_utils import execute_sparql_update
from graphiti_core.driver.oracle_driver import OracleDriver
from graphiti_core.edges import EntityEdge
from graphiti_core.models.edges.edge_db_queries import get_entity_edge_save_query
from graphiti_core.models.nodes.node_db_queries import get_entity_node_save_query
from graphiti_core.nodes import EntityNode
logging.getLogger('graphiti_core.driver.oracle_driver').setLevel(logging.INFO)

In [ ]:
def _extract_single_count(rows: list[dict]) -> int:
    if not rows:
        raise AssertionError('Expected at least one row from COUNT query.')
    first_row = rows[0]
    if not first_row:
        raise AssertionError('COUNT query returned an empty row.')
    return int(next(iter(first_row.values())))


async def run_oracle_rdf_end_to_end(driver: OracleDriver) -> dict:
    """Run a manual Oracle RDF end-to-end verification suite with a provided driver."""
    if not isinstance(driver, OracleDriver):
        raise TypeError('driver must be an OracleDriver instance.')

    if not driver.rdf_enabled:
        raise RuntimeError(
            'This notebook requires Oracle RDF mode. Pass a driver with use_rdf=True '
            'or set ORACLE_USE_RDF=true when creating your driver.'
        )

    if driver.entity_node_ops is None or driver.entity_edge_ops is None:
        raise RuntimeError('Oracle driver operation namespaces are not initialized.')

    group_id = f'notebook-suite-{uuid4().hex[:8]}'
    created_at = datetime.now(timezone.utc)
    marker_subject = f'<urn:graphiti:notebook:marker:{uuid4().hex}>'

    rdf_table_name = driver.rdf_table_name()
    rdf_owner, rdf_table = rdf_table_name.split('.', maxsplit=1)

    # Driver smoke + pool reuse check.
    connection_rows, _, _ = await driver.execute_query('SELECT 1 AS value FROM dual')
    pool_before = driver.client
    await driver.execute_query('SELECT 1 AS value FROM dual')
    pool_after = driver.client

    # Query builder checks aligned with tests/driver/test_oracle_queries.py.
    node_query = get_entity_node_save_query(GraphProvider.ORACLE, 'Entity:Person')
    edge_query = get_entity_edge_save_query(GraphProvider.ORACLE)

    builder_checks = {
        'node_query_avoids_neo4j_vector_procedure': 'db.create.setNodeVectorProperty'
        not in node_query,
        'edge_query_avoids_neo4j_vector_procedure': 'db.create.setRelationshipVectorProperty'
        not in edge_query,
    }

    before_rows, _, _ = await driver.execute_query(
        f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}'
    )
    triple_count_before = _extract_single_count(before_rows)

    table_rows, _, _ = await driver.execute_query(
        """
        SELECT owner, table_name, tablespace_name
        FROM all_tables
        WHERE owner = $owner AND table_name = $table_name
        """,
        owner=rdf_owner,
        table_name=rdf_table,
    )

    alice = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Alice',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Notebook integration test entity (Alice).',
        attributes={'role': 'engineer', 'source': 'tests/test_oracle_rdf_end_to_end.ipynb'},
    )
    bob = EntityNode(
        uuid=f'entity-{uuid4().hex[:8]}',
        name='Bob',
        group_id=group_id,
        labels=['Person'],
        created_at=created_at,
        summary='Notebook integration test entity (Bob).',
        attributes={'role': 'manager', 'source': 'tests/test_oracle_rdf_end_to_end.ipynb'},
    )
    works_with = EntityEdge(
        uuid=f'edge-{uuid4().hex[:8]}',
        group_id=group_id,
        source_node_uuid=alice.uuid,
        target_node_uuid=bob.uuid,
        created_at=created_at,
        name='works_with',
        fact='Alice works with Bob in Oracle RDF notebook verification.',
        episodes=[],
        valid_at=created_at,
        attributes={'confidence': 0.98, 'source': 'tests/test_oracle_rdf_end_to_end.ipynb'},
    )

    cleanup_errors: list[str] = []
    after_insert_count = triple_count_before

    try:
        # Direct RDF utils path to verify driver-level RDF identifiers flow through.
        await execute_sparql_update(
            driver,
            f'DELETE WHERE {{ {marker_subject} ?p ?o . }}; '
            f'INSERT DATA {{ {marker_subject} <urn:graphiti:pred:type> "NOTEBOOK_MARKER" . }}',
        )

        await driver.entity_node_ops.save(driver, alice)
        await driver.entity_node_ops.save(driver, bob)
        await driver.entity_edge_ops.save(driver, works_with)

        after_rows, _, _ = await driver.execute_query(
            f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}'
        )
        after_insert_count = _extract_single_count(after_rows)
    finally:
        for cleanup_label, cleanup_call in [
            (
                'delete_entity_edge',
                driver.entity_edge_ops.delete_by_uuids(driver, [works_with.uuid]),
            ),
            (
                'delete_group_nodes_and_edges',
                driver.entity_node_ops.delete_by_group_id(driver, group_id),
            ),
            (
                'delete_notebook_marker',
                execute_sparql_update(driver, f'DELETE WHERE {{ {marker_subject} ?p ?o . }}'),
            ),
        ]:
            try:
                await cleanup_call
            except Exception as exc:
                cleanup_errors.append(f'{cleanup_label}: {exc}')

    final_rows, _, _ = await driver.execute_query(
        f'SELECT COUNT(*) AS triple_count FROM {rdf_table_name}'
    )
    triple_count_after_cleanup = _extract_single_count(final_rows)

    return {
        'connection_rows': connection_rows,
        'pool_reused_across_calls': pool_before is not None and pool_before is pool_after,
        'builder_checks': builder_checks,
        'rdf_table_name': rdf_table_name,
        'rdf_table_metadata_rows': table_rows,
        'rdf_ids_seen_by_driver': {
            'network_owner': driver.rdf_network_owner,
            'network_name': driver.rdf_network_name,
            'graph_name': driver.rdf_graph_name,
        },
        'created_group_id': group_id,
        'created_entities': [alice.uuid, bob.uuid],
        'created_edge': works_with.uuid,
        'triple_count_before': triple_count_before,
        'triple_count_after_insert': after_insert_count,
        'triple_count_after_cleanup': triple_count_after_cleanup,
        'cleanup_errors': cleanup_errors,
    }

In [ ]:
# Provide your initialized OracleDriver here.
#
# Example:
# driver = OracleDriver(
#     dsn='host:1521/service_name',
#     user='scott',
#     password='tiger',
#     use_rdf=True,
#     rdf_network_owner='RDFUSER',
#     rdf_network_name='NET1',
#     rdf_graph_name='GRAPHITI',
#     log_queries=True,
# )

driver = None

In [ ]:
if driver is None:
    raise RuntimeError('Set `driver` in the previous cell before running this test suite.')

results = await run_oracle_rdf_end_to_end(driver)
pprint(results)

if results['cleanup_errors']:
    raise RuntimeError(f"Cleanup encountered errors: {results['cleanup_errors']}")

if not results['pool_reused_across_calls']:
    raise RuntimeError('Expected Oracle driver to reuse the same pool across requests.')

if not all(results['builder_checks'].values()):
    raise RuntimeError(f"Query-builder checks failed: {results['builder_checks']}")

print('\nOracle RDF notebook verification completed successfully.')